In [1]:
import os
os.chdir(r"C:\JupyterProjects\Stock_ML_Project")  # <- adjust to your path
print(os.getcwd())

C:\JupyterProjects\Stock_ML_Project


In [2]:
# 06_classification_baseline_all_stocks.ipynb
# Paste and run in a new notebook.

# -----------------------
# Imports & helpers
# -----------------------
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

pd.options.display.float_format = '{:,.6f}'.format

# -----------------------
# Config
# -----------------------
PROJECT_ROOT = Path('.')  # change to project root if notebook runs from subfolder
processed_dir = PROJECT_ROOT / "data" / "processed"
figures_clf_dir = PROJECT_ROOT / "figures" / "models" / "classification"
results_dir = PROJECT_ROOT / "results"

for p in [figures_clf_dir, results_dir]:
    p.mkdir(parents=True, exist_ok=True)

files = {
    "RELIANCE": processed_dir / "reliance_model_ready.csv",
    "TCS": processed_dir / "tcs_model_ready.csv",
    "HDFCBANK": processed_dir / "hdfcbank_model_ready.csv"
}

# Classifiers to evaluate
classifiers = {
    "LogisticRegression": LogisticRegression(max_iter=1000, solver='liblinear'),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVC_rbf": SVC(kernel='rbf', probability=True, gamma='scale')
}

# -----------------------
# Utility functions
# -----------------------
def safe_find(col_list, substr):
    for c in col_list:
        if substr in c:
            return c
    return None

def plot_and_save_confusion(cm, labels, title, outpath):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# -----------------------
# Main loop over stocks
# -----------------------
all_results = []

for ticker, filepath in files.items():
    print(f"\n=== Processing {ticker} -> {filepath} ===")
    if not filepath.exists():
        print("  WARNING: file not found, skipping:", filepath)
        continue

    df = pd.read_csv(filepath)
    print("  Loaded shape:", df.shape)

    # detect columns
    target_cls_col = safe_find(df.columns, "Target_Cls") or safe_find(df.columns, "Target") or safe_find(df.columns, "target_cls")
    split_col = safe_find(df.columns, "Split") or safe_find(df.columns, "split")

    if target_cls_col is None or split_col is None:
        print("  ERROR: couldn't detect Target_Cls or Split column. Columns:", list(df.columns))
        continue

    print("  Using classification target:", target_cls_col, ", split column:", split_col)

    # prepare features and labels
    # drop the target cols and split; keep everything else as X
    drop_cols = [target_cls_col, "Target_Reg", "Split"]
    drop_cols = [c for c in drop_cols if c in df.columns]
    X = df.drop(columns=drop_cols)
    y = df[target_cls_col]

    # chronological split via Split column (case-insensitive)
    train_mask = df[split_col].astype(str).str.lower() == "train"
    test_mask  = df[split_col].astype(str).str.lower() == "test"

    X_train = X[train_mask].reset_index(drop=True)
    X_test  = X[test_mask].reset_index(drop=True)
    y_train = y[train_mask].reset_index(drop=True)
    y_test  = y[test_mask].reset_index(drop=True)

    print(f"  Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
    if X_train.shape[0] == 0 or X_test.shape[0] == 0:
        print("  ERROR: empty train/test; skipping.")
        continue

    # create folder per ticker for plots
    ticker_fig_dir = figures_clf_dir / ticker
    ticker_fig_dir.mkdir(parents=True, exist_ok=True)

    # For scaling when needed (KNN, SVC)
    scaler = StandardScaler()
    scaler.fit(X_train)  # fit on train

    for model_name, model_obj in classifiers.items():
        # fresh instance
        ModelClass = model_obj.__class__
        params = model_obj.get_params()
        clf = ModelClass(**params)

        # If model benefits from scaling, use scaled versions
        needs_scale = model_name.lower().startswith("knn") or model_name.lower().startswith("svc")
        if needs_scale:
            X_train_in = scaler.transform(X_train)
            X_test_in  = scaler.transform(X_test)
        else:
            X_train_in = X_train.values
            X_test_in  = X_test.values

        # fit and predict
        clf.fit(X_train_in, y_train)
        y_pred = clf.predict(X_test_in)

        # predicted probabilities if available (for ROC-AUC)
        y_prob = None
        try:
            if hasattr(clf, "predict_proba"):
                y_prob = clf.predict_proba(X_test_in)[:,1]
            elif hasattr(clf, "decision_function"):
                # convert decision scores with sigmoid-like approach (not exactly prob)
                scores = clf.decision_function(X_test_in)
                # optional: min-max normalize to 0-1 for a crude AUC input (but roc_auc accepts scores too)
                y_prob = scores
        except Exception:
            y_prob = None

        # metrics
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        roc_auc = None
        if y_prob is not None:
            try:
                roc_auc = roc_auc_score(y_test, y_prob)
            except Exception:
                roc_auc = None

        # directional accuracy = accuracy in this up/down task
        directional_acc = acc

        # append results
        all_results.append({
            "Ticker": ticker,
            "Model": model_name,
            "Accuracy": acc,
            "Precision": prec,
            "Recall": rec,
            "F1": f1,
            "ROC_AUC": roc_auc if roc_auc is not None else np.nan,
            "Directional_Acc": directional_acc,
            "TrainRows": X_train.shape[0],
            "TestRows": X_test.shape[0]
        })

        # save preds & probs
        preds_df = pd.DataFrame({"Actual": y_test.values, "Predicted": y_pred})
        if y_prob is not None:
            preds_df["Prob"] = y_prob
        preds_csv = results_dir / f"{ticker}_{model_name}_preds.csv"
        preds_df.to_csv(preds_csv, index=False)

        # confusion matrix plot
        cm = confusion_matrix(y_test, y_pred)
        cm_path = ticker_fig_dir / f"{ticker}_{model_name}_confmat.png"
        plot_and_save_confusion(cm, labels=[0,1], title=f"{ticker} {model_name} Confusion Matrix", outpath=cm_path)

        print(f"   -> {model_name}: Acc={acc:.4f}, F1={f1:.4f}, ROC_AUC={roc_auc}")

    # optional: quick summary plot of accuracy across models for this ticker
    df_ticker_res = pd.DataFrame([r for r in all_results if r["Ticker"]==ticker])
    if not df_ticker_res.empty:
        plt.figure(figsize=(8,4))
        sns.barplot(x="Model", y="Accuracy", data=df_ticker_res.sort_values("Accuracy", ascending=False))
        plt.xticks(rotation=30)
        plt.title(f"{ticker} - Model Accuracy")
        plt.tight_layout()
        plt.savefig(ticker_fig_dir / f"{ticker}_accuracy_bar.png", dpi=200)
        plt.close()

# Save combined results
results_df = pd.DataFrame(all_results)
results_csv = results_dir / "classification_results.csv"
results_df.to_csv(results_csv, index=False)
print("\nCompleted. Classification summary saved to:", results_csv)
display(results_df.sort_values(["Ticker","Model"]))



=== Processing RELIANCE -> data\processed\reliance_model_ready.csv ===
  Loaded shape: (1460, 9)
  Using classification target: Target_Cls , split column: Split
  Train size: 1168, Test size: 292
   -> LogisticRegression: Acc=0.5034, F1=0.5797, ROC_AUC=0.4977631269131151
   -> KNN: Acc=0.5000, F1=0.4427, ROC_AUC=0.5323522486461032
   -> DecisionTree: Acc=0.4897, F1=0.1183, ROC_AUC=0.5176595243701436
   -> RandomForest: Acc=0.4932, F1=0.1778, ROC_AUC=0.5662114433717919
   -> SVC_rbf: Acc=0.5171, F1=0.3502, ROC_AUC=0.5404285377913822

=== Processing TCS -> data\processed\tcs_model_ready.csv ===
  Loaded shape: (1460, 9)
  Using classification target: Target_Cls , split column: Split
  Train size: 1168, Test size: 292
   -> LogisticRegression: Acc=0.5342, F1=0.5467, ROC_AUC=0.5159065315315315
   -> KNN: Acc=0.5205, F1=0.4400, ROC_AUC=0.5068036786786787
   -> DecisionTree: Acc=0.5205, F1=0.2045, ROC_AUC=0.5152027027027027
   -> RandomForest: Acc=0.5240, F1=0.1677, ROC_AUC=0.49685623123123

,Ticker,Model,Accuracy,Precision,Recall,F1,ROC_AUC,Directional_Acc,TrainRows,TestRows
12,HDFCBANK,DecisionTree,0.479452,0.513699,0.480769,0.496689,0.479355,0.479452,1168,292
11,HDFCBANK,KNN,0.561644,0.612903,0.487179,0.542857,0.558776,0.561644,1168,292
10,HDFCBANK,LogisticRegression,0.506849,0.535294,0.583333,0.558282,0.519702,0.506849,1168,292
13,HDFCBANK,RandomForest,0.534247,0.598039,0.391026,0.472868,0.525995,0.534247,1168,292
14,HDFCBANK,SVC_rbf,0.469178,0.502618,0.615385,0.553314,0.520857,0.469178,1168,292
2,RELIANCE,DecisionTree,0.489726,0.714286,0.064516,0.118343,0.517660,0.489726,1168,292
1,RELIANCE,KNN,0.500000,0.542056,0.374194,0.442748,0.532352,0.500000,1168,292
0,RELIANCE,LogisticRegression,0.503425,0.526316,0.645161,0.579710,0.497763,0.503425,1168,292
3,RELIANCE,RandomForest,0.493151,0.640000,0.103226,0.177778,0.566211,0.493151,1168,292
4,RELIANCE,SVC_rbf,0.517123,0.612903,0.245161,0.350230,0.540429,0.517123,1168,292
